In [22]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

# 1 load the data
CSV_PATH = "spending_l9_dataset.csv"
df = pd.read_csv(CSV_PATH)
# print(df.head())

# 2 preparing features
FEATURES = ["Income_$", "SpendingScore"]
X = df[FEATURES].copy()

# Handle missing values with median
for col in FEATURES:
    if X[col].isna().any():
        X[col] = X[col].fillna(X[col].median())

# print("\n FEATURES")
print(X.head())

# 3 scaling the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4) Elbow Method (SSE)
print("\n=== ELBOW METHOD (SSE per k) ===")

for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init="auto")
    km.fit(X_scaled)
    print(f"k={k} → SSE={km.inertia_:.2f}")

    # 5)  Model training

kmeans = KMeans(n_clusters=4, random_state=42, n_init="auto")
labels = kmeans.fit_predict(X_scaled)
df["Cluster"] = labels.astype(int)
print("\n SAMPLE DATA WITH CLUSTERS")
print(df.head())

# 6) Evaluate Clustering
sil = silhouette_score(X_scaled, labels)
dbi = davies_bouldin_score(X_scaled, labels)

print("\n=== CLUSTER METRICS ===")
print(f"Silhouette Score : {sil:.3f}")
print(f"Davies–Bouldin   : {dbi:.3f}")

# 7) Cluster Centers
centers_scaled = kmeans.cluster_centers_
centers_original = scaler.inverse_transform(centers_scaled)

centers_df = pd.DataFrame(centers_original, columns=FEATURES)
centers_df.index.name = "Cluster"
print("Cluster Centers:")
print(centers_df.round(2))

# 8) Sanity Check
sample_rows = df.loc[[0, 1, 2], ["Income_$", "SpendingScore", "Cluster"]]
print(sample_rows)

# 9) Save Output
OUT_FILE = "spending_labeled_clusters.csv"
df.to_csv(OUT_FILE, index=False)

print(f"\nSaved labeled dataset → {OUT_FILE}")

   Income_$  SpendingScore
0        33             78
1        25             87
2        24             88
3        25             73
4        23             88

=== ELBOW METHOD (SSE per k) ===
k=1 → SSE=400.00
k=2 → SSE=199.70
k=3 → SSE=79.37
k=4 → SSE=21.37
k=5 → SSE=19.09
k=6 → SSE=15.65
k=7 → SSE=14.48
k=8 → SSE=13.81
k=9 → SSE=12.94
k=10 → SSE=11.52

 SAMPLE DATA WITH CLUSTERS
   CustomerID  Age  Income_$  SpendingScore  VisitsPerMonth  OnlinePurchases  \
0           1   28        33             78              14                9   
1           2   21        25             87               8               23   
2           3   23        24             88              13               10   
3           4   24        25             73              16               11   
4           5   20        23             88              17               16   

   Gender Region  Cluster  
0  Female   East        2  
1    Male  North        2  
2    Male  South        2  
3  Female   West    